# YOLOv2 Traffic Detection Experiment — IB Extended Essay

**RQ:** To what extent does changing the parameters of the YOLO (You Only
Look Once) algorithm affect the decisions made by Apollo Go robotaxis to
safely handle traffic scenarios?

This notebook is written in a plain, step-by-step style on purpose — no
custom functions, no fancy shortcuts — just straightforward code from top
to bottom, the way you'd actually type it out yourself. Some steps repeat
the same handful of lines more than once instead of packaging them away;
that's intentional, so every cell can be read start to finish on its own.

**How to use this notebook:** run the cells in order, one at a time
(click a cell, press `Shift + Enter`). Read the comments — they explain
what every line is doing, which you'll need for your essay's methodology
section.


## Step 1 — Install the library we need

We only need to add OpenCV. Colab already has numpy and matplotlib, so we leave those alone.

In [ ]:
# pip is the tool that installs extra code libraries for Python.
# opencv-python-headless is OpenCV - it's what lets us open images and
# run the YOLO neural network. --quiet just hides the long install log.
!pip install --quiet opencv-python-headless


## Step 2 — Create the folders and the YOLOv2 config files

YOLOv2 needs two small text files:
- `yolov2.cfg` — describes the shape of the neural network
- `coco.names` — the list of 80 object names YOLOv2 knows (e.g. "person", "car", "bicycle")

Both are small plain text, so this cell just writes them straight to disk instead of downloading them.

In [ ]:
import os

# create the folders we'll use, if they don't already exist
os.makedirs("model", exist_ok=True)
os.makedirs("sample_images", exist_ok=True)
os.makedirs("results", exist_ok=True)

yolov2_cfg_text = '[net]\n# Testing\nbatch=1\nsubdivisions=1\n# Training\n# batch=64\n# subdivisions=8\nwidth=608\nheight=608\nchannels=3\nmomentum=0.9\ndecay=0.0005\nangle=0\nsaturation = 1.5\nexposure = 1.5\nhue=.1\n\nlearning_rate=0.001\nburn_in=1000\nmax_batches = 500200\npolicy=steps\nsteps=400000,450000\nscales=.1,.1\n\n[convolutional]\nbatch_normalize=1\nfilters=32\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=64\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=64\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=128\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=256\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[maxpool]\nsize=2\nstride=2\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=512\nsize=1\nstride=1\npad=1\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nfilters=1024\nsize=3\nstride=1\npad=1\nactivation=leaky\n\n\n#######\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[route]\nlayers=-9\n\n[convolutional]\nbatch_normalize=1\nsize=1\nstride=1\npad=1\nfilters=64\nactivation=leaky\n\n[reorg]\nstride=2\n\n[route]\nlayers=-1,-4\n\n[convolutional]\nbatch_normalize=1\nsize=3\nstride=1\npad=1\nfilters=1024\nactivation=leaky\n\n[convolutional]\nsize=1\nstride=1\npad=1\nfilters=425\nactivation=linear\n\n\n[region]\nanchors =  0.57273, 0.677385, 1.87446, 2.06253, 3.33843, 5.47434, 7.88282, 3.52778, 9.77052, 9.16828\nbias_match=1\nclasses=80\ncoords=4\nnum=5\nsoftmax=1\njitter=.3\nrescore=1\n\nobject_scale=5\nnoobject_scale=1\nclass_scale=1\ncoord_scale=1\n\nabsolute=1\nthresh = .6\nrandom=1\n'
cfg_file = open("model/yolov2.cfg", "w")
cfg_file.write(yolov2_cfg_text)
cfg_file.close()

coco_names_text = 'person\nbicycle\ncar\nmotorbike\naeroplane\nbus\ntrain\ntruck\nboat\ntraffic light\nfire hydrant\nstop sign\nparking meter\nbench\nbird\ncat\ndog\nhorse\nsheep\ncow\nelephant\nbear\nzebra\ngiraffe\nbackpack\numbrella\nhandbag\ntie\nsuitcase\nfrisbee\nskis\nsnowboard\nsports ball\nkite\nbaseball bat\nbaseball glove\nskateboard\nsurfboard\ntennis racket\nbottle\nwine glass\ncup\nfork\nknife\nspoon\nbowl\nbanana\napple\nsandwich\norange\nbroccoli\ncarrot\nhot dog\npizza\ndonut\ncake\nchair\nsofa\npottedplant\nbed\ndiningtable\ntoilet\ntvmonitor\nlaptop\nmouse\nremote\nkeyboard\ncell phone\nmicrowave\noven\ntoaster\nsink\nrefrigerator\nbook\nclock\nvase\nscissors\nteddy bear\nhair drier\ntoothbrush\n'
names_file = open("model/coco.names", "w")
names_file.write(coco_names_text)
names_file.close()

print("Wrote model/yolov2.cfg and model/coco.names")


## Step 3 — Download the YOLOv2 weights file (~194 MB)

This is YOLOv2's trained "knowledge" - too big to type out like the files above, so we download it instead. Can take a couple of minutes.

**If this fails**, tell your helper (Claude) and we'll find a different download link.

In [ ]:
import os

if os.path.exists("model/yolov2.weights") and os.path.getsize("model/yolov2.weights") > 100_000_000:
    print("Already downloaded, skipping.")
else:
    !curl -L -o model/yolov2.weights https://pjreddie.com/media/files/yolov2.weights
    !ls -lh model/yolov2.weights


## Step 4 — Upload your traffic photo

Do this part by hand: on the left side of Colab, click the folder icon to open the file browser, open the `sample_images` folder, click the upload icon, and pick your traffic photo. Then rename it (right-click the file -> Rename) to exactly `traffic_scene_1.jpg`.

**Where to get a photo:** a frame from a real Apollo Go ride-along video (screenshot it - note the video title/URL/timestamp for your citation), a photo from an open self-driving dataset (e.g. BDD100K), or a free-to-use street photo (Wikimedia Commons, Pexels) - keep the source link for your bibliography.

Once it's uploaded and renamed, this cell just tells the rest of the notebook where to find it:

In [ ]:
# the path to the photo you just uploaded into sample_images/
IMAGE_PATH = "sample_images/traffic_scene_1.jpg"


## Step 5 — Load YOLOv2 and the 80 object names

We only need to do this once - the rest of the notebook reuses `net` (the loaded network) and `class_names` (the list of names) from here.

In [ ]:
import cv2
import numpy as np

# open coco.names and turn it into a plain list, one name per line
names_file = open("model/coco.names", "r")
file_text = names_file.read()
names_file.close()

class_names = file_text.strip().split("\n")
print("Loaded", len(class_names), "class names, for example:", class_names[0], class_names[1], class_names[2])

# load the YOLO network: yolov2.cfg is the "blueprint", yolov2.weights is
# the trained "knowledge". This can take a few seconds.
net = cv2.dnn.readNetFromDarknet("model/yolov2.cfg", "model/yolov2.weights")
print("YOLOv2 network loaded")


## Step 6 — Sanity check: run YOLO once on your photo

This is the core detection code, written out in full so you can see exactly what happens, step by step, with normal/default settings. The experiment in Step 7 does the exact same thing, just repeated with different `CONFIDENCE_THRESHOLD` values.

In [ ]:
from IPython.display import Image, display

# --- these 3 numbers are the parameters my RQ investigates ---
CONFIDENCE_THRESHOLD = 0.5   # how sure YOLO must be (0-1) to keep a detection
NMS_THRESHOLD = 0.4          # how strictly overlapping boxes get merged into one
INPUT_SIZE = 416              # the width/height (pixels) YOLO resizes the photo to

# --- load the photo ---
image = cv2.imread(IMAGE_PATH)
image_height = image.shape[0]
image_width = image.shape[1]

# --- turn the photo into the special format ("blob") YOLO needs ---
# resizes it to INPUT_SIZE x INPUT_SIZE, scales colours from 0-255 down to
# 0.0-1.0, and swaps colour order from OpenCV's BGR to YOLO's RGB
blob = cv2.dnn.blobFromImage(image, 1 / 255.0, (INPUT_SIZE, INPUT_SIZE), swapRB=True, crop=False)
net.setInput(blob)

# --- find the names of the network's final output layers ---
layer_names = net.getLayerNames()
output_layer_numbers = net.getUnconnectedOutLayers()
output_layer_names = []
for number in output_layer_numbers:
    output_layer_names.append(layer_names[number - 1])

# --- run the network - this is the actual "looking at the photo" step ---
layer_outputs = net.forward(output_layer_names)

# these three lists will hold what we decide to keep
boxes = []
confidences = []
class_ids = []

# go through every single box YOLO guessed at
for output in layer_outputs:
    for detection in output:
        # the first 5 numbers are box position info; the other 80 numbers
        # are a confidence score for each of the 80 possible object classes
        scores = detection[5:]

        # find which class has the highest score, using a simple loop
        # instead of a built-in shortcut, so it's clear what's happening
        best_score = 0
        best_class_id = 0
        for i in range(len(scores)):
            if scores[i] > best_score:
                best_score = scores[i]
                best_class_id = i

        # only keep this box if YOLO is confident enough - this is
        # exactly where CONFIDENCE_THRESHOLD gets used
        if best_score > CONFIDENCE_THRESHOLD:
            # box position comes as a FRACTION of the image size (0-1),
            # so multiply by the real width/height to get pixels
            center_x = int(detection[0] * image_width)
            center_y = int(detection[1] * image_height)
            box_width = int(detection[2] * image_width)
            box_height = int(detection[3] * image_height)
            # OpenCV wants the top-left corner, not the centre
            x = int(center_x - box_width / 2)
            y = int(center_y - box_height / 2)

            boxes.append([x, y, box_width, box_height])
            confidences.append(float(best_score))
            class_ids.append(best_class_id)

# --- remove duplicate/overlapping boxes on the same object ---
# this is exactly where NMS_THRESHOLD gets used
boxes_to_keep = cv2.dnn.NMSBoxes(boxes, confidences, CONFIDENCE_THRESHOLD, NMS_THRESHOLD)

print("YOLO found", len(boxes_to_keep), "object(s):")
for i in boxes_to_keep.flatten():
    name = class_names[class_ids[i]]
    percent_sure = round(confidences[i] * 100)
    print("-", name, "-", percent_sure, "% sure")

# --- draw a box and label on the photo for every object we kept ---
labelled_image = image.copy()
for i in boxes_to_keep.flatten():
    x, y, w, h = boxes[i]
    label_text = class_names[class_ids[i]] + " " + str(round(confidences[i] * 100)) + "%"
    cv2.rectangle(labelled_image, (x, y), (x + w, y + h), (0, 255, 0), 2)
    cv2.putText(labelled_image, label_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

cv2.imwrite("results/sanity_check.jpg", labelled_image)
display(Image("results/sanity_check.jpg"))


## Step 7 — The actual experiment: vary the confidence threshold

This is the part that produces the data for the essay. We run the exact same steps as Step 6, but inside a loop that tries 9 different `CONFIDENCE_THRESHOLD` values, one at a time, on the *same* photo (everything else stays the same - that's what makes it a controlled experiment). For each one we count how many objects were found in total, and how many were "safety-critical" (person, car, bicycle, traffic light, etc - the object types that would actually change a robotaxi's driving decision).

In [ ]:
import csv

# the values we are testing - this is the independent variable
confidence_values_to_test = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

# kept constant across every test - the controlled variables
FIXED_NMS_THRESHOLD = 0.4
FIXED_INPUT_SIZE = 416

# object types that matter for a robotaxi's safety decisions
safety_classes = ["person", "bicycle", "car", "motorbike", "bus", "truck",
                   "traffic light", "stop sign"]

# these lists will collect one result per confidence value tested
tested_thresholds = []
total_objects_list = []
safety_objects_list = []

original_image = cv2.imread(IMAGE_PATH)
image_height = original_image.shape[0]
image_width = original_image.shape[1]

for confidence_value in confidence_values_to_test:
    print("Testing confidence threshold:", confidence_value)

    # --- same blob + forward pass steps as Step 6 ---
    blob = cv2.dnn.blobFromImage(original_image, 1 / 255.0,
                                  (FIXED_INPUT_SIZE, FIXED_INPUT_SIZE),
                                  swapRB=True, crop=False)
    net.setInput(blob)
    layer_names = net.getLayerNames()
    output_layer_numbers = net.getUnconnectedOutLayers()
    output_layer_names = []
    for number in output_layer_numbers:
        output_layer_names.append(layer_names[number - 1])
    layer_outputs = net.forward(output_layer_names)

    boxes = []
    confidences = []
    class_ids = []

    for output in layer_outputs:
        for detection in output:
            scores = detection[5:]
            best_score = 0
            best_class_id = 0
            for i in range(len(scores)):
                if scores[i] > best_score:
                    best_score = scores[i]
                    best_class_id = i

            if best_score > confidence_value:
                center_x = int(detection[0] * image_width)
                center_y = int(detection[1] * image_height)
                box_width = int(detection[2] * image_width)
                box_height = int(detection[3] * image_height)
                x = int(center_x - box_width / 2)
                y = int(center_y - box_height / 2)
                boxes.append([x, y, box_width, box_height])
                confidences.append(float(best_score))
                class_ids.append(best_class_id)

    boxes_to_keep = cv2.dnn.NMSBoxes(boxes, confidences, confidence_value, FIXED_NMS_THRESHOLD)

    # --- count total objects and safety-critical objects ---
    total_count = 0
    safety_count = 0
    for i in boxes_to_keep.flatten():
        total_count = total_count + 1
        if class_names[class_ids[i]] in safety_classes:
            safety_count = safety_count + 1

    tested_thresholds.append(confidence_value)
    total_objects_list.append(total_count)
    safety_objects_list.append(safety_count)

    # --- save a labelled photo for this threshold ---
    labelled_image = original_image.copy()
    for i in boxes_to_keep.flatten():
        x, y, w, h = boxes[i]
        label_text = class_names[class_ids[i]] + " " + str(round(confidences[i] * 100)) + "%"
        cv2.rectangle(labelled_image, (x, y), (x + w, y + h), (0, 255, 0), 2)
        cv2.putText(labelled_image, label_text, (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
    image_filename = "results/detected_conf_" + str(confidence_value) + ".jpg"
    cv2.imwrite(image_filename, labelled_image)

    print("  total objects:", total_count, "| safety-critical objects:", safety_count)

# --- save the results as a CSV table for the essay ---
csv_file = open("results/experiment_results.csv", "w", newline="")
writer = csv.writer(csv_file)
writer.writerow(["confidence_threshold", "total_objects", "safety_critical_objects"])
for i in range(len(tested_thresholds)):
    writer.writerow([tested_thresholds[i], total_objects_list[i], safety_objects_list[i]])
csv_file.close()

print("\nDone! Saved results/experiment_results.csv")


## Step 8 — Look at the results table and chart

In [ ]:
# print a simple table straight from the lists we already have
print("threshold | total objects | safety-critical objects")
for i in range(len(tested_thresholds)):
    print(tested_thresholds[i], "     |", total_objects_list[i], "            |", safety_objects_list[i])

# (the same numbers are also saved in results/experiment_results.csv,
# which you can open in Excel/Google Sheets to make a neater table
# for your essay)


In [ ]:
import matplotlib.pyplot as plt

x_positions = range(len(tested_thresholds))
bar_width = 0.35

plt.figure(figsize=(8, 5))
plt.bar([x - bar_width / 2 for x in x_positions], total_objects_list,
        width=bar_width, label="All objects detected")
plt.bar([x + bar_width / 2 for x in x_positions], safety_objects_list,
        width=bar_width, label="Safety-critical objects detected")

plt.xticks(list(x_positions), [str(t) for t in tested_thresholds])
plt.xlabel("Confidence threshold")
plt.ylabel("Number of objects detected")
plt.title("Effect of YOLOv2 confidence threshold on detections")
plt.legend()
plt.tight_layout()
plt.savefig("results/confidence_vs_detections.png")
plt.show()


See objects appearing/disappearing as the threshold changes by comparing a low-threshold photo against a high-threshold one:

In [ ]:
from IPython.display import Image, display

display(Image("results/detected_conf_0.1.jpg"))  # low threshold: more detections kept
display(Image("results/detected_conf_0.9.jpg"))  # high threshold: fewer, more certain detections


## Step 9 — Download everything for your essay

In [ ]:
from google.colab import files

!zip -rq results.zip results
files.download("results.zip")


---
## Using these results in your essay

See the "4. Using these results in the essay" section of this project's
`README.md` for:
- an Independent/Dependent/Controlled variables table for your
  methodology section
- a 4-step structure for turning these numbers into an argument
  (state the pattern → connect it to a driving decision → extend to
  Apollo Go → state limitations)
- a fill-in-the-numbers example paragraph

**Quick summary of what each parameter means, in case you need it while writing:**
- **`CONFIDENCE_THRESHOLD`** — how sure YOLO must be before it reports a
  detection at all. Controls false negatives (missed hazards) vs false
  positives (phantom detections).
- **`NMS_THRESHOLD`** — how aggressively overlapping boxes on the same
  object get merged into one. Mostly about not double-counting the same
  object.
- **`INPUT_SIZE`** — the resolution YOLO actually "looks" at. Bigger =
  better at spotting small/far-away objects, but slower - which matters
  for a moving car needing real-time answers.
